# 📊 Financial Analysis Platform
## Macro & Micro Analysis — Investment Decision Support

**Universo de activos cubierto:**
- **Acciones individuales:** NVIDIA (NVDA), Microsoft (MSFT), Alphabet (GOOGL)
- **ETFs Temáticos:** Microprocesadores (SOXX), Semiconductores (SMH), Energía Solar (TAN), Energía Nuclear (NLR/URNM)
- **Índices:** S&P 500 (^GSPC / SPY), NASDAQ Composite (^IXIC / QQQ)

**Módulos implementados:**
1. `Módulo A` — Macroeconómico (Yield Curve, Tasas Fed, Inflación CPI)
2. `Módulo B` — Microeconómico / Fundamental (Ratios, Financieros, Earnings)
3. `Módulo C` — Mercado / Técnico (Precios OHLCV, Indicadores, Volatilidad)
4. `Módulo D` — Portafolio / Screening (Análisis, Optimización Markowitz)

---
## 🔧 0. Instalación de Dependencias

In [ ]:
# ── Instalación silenciosa ──────────────────────────────────────────────
import subprocess, sys

packages = [
    'yfinance>=0.2.40',
    'pandas>=2.0',
    'numpy>=1.26',
    'matplotlib>=3.8',
    'seaborn>=0.13',
    'plotly>=5.20',
    'scipy>=1.12',
    'scikit-learn>=1.4',
    'PyPortfolioOpt>=1.5.5',
    'pandas-datareader>=0.10',
    'requests>=2.31',
    'ta>=0.11',           # Technical Analysis library (RSI, MACD, BBANDS, ATR)
    'statsmodels>=0.14',  # Regresiones, CAPM
    'pyfolio-reloaded',   # Tearsheet de portafolios
    'ipywidgets',
]

for pkg in packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True
    )

print('✅ Dependencias instaladas correctamente.')

---
## 📦 1. Importaciones y Configuración Global

In [ ]:
# ── Core ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import requests
import json

# ── Datos financieros ───────────────────────────────────────────────────
import yfinance as yf
import pandas_datareader.data as web

# ── Análisis técnico ────────────────────────────────────────────────────
import ta
from ta.trend import MACD, SMAIndicator, EMAIndicator, ADXIndicator
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

# ── Estadísticas y ML ───────────────────────────────────────────────────
from scipy import stats
from scipy.optimize import minimize
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Optimización de portafolio ──────────────────────────────────────────
from pypfopt import EfficientFrontier, risk_models, expected_returns
from pypfopt import plotting as pf_plotting
from pypfopt.discrete_allocation import DiscreteAllocation
from pypfopt import BlackLittermanModel, objective_functions

# ── Visualización ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Configuración global de estilo ──────────────────────────────────────
plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

COLORS = {
    'NVDA':  '#76b900', 'MSFT':  '#00a4ef', 'GOOGL': '#fbbc05',
    'SOXX':  '#ff6b35', 'SMH':   '#f7c59f', 'TAN':   '#f9d71c',
    'NLR':   '#8b0000', 'URNM':  '#dc143c', 'SPY':   '#1f77b4',
    'QQQ':   '#9467bd',
}

print('✅ Entorno configurado.')
print(f'   pandas  {pd.__version__} | numpy {np.__version__} | yfinance {yf.__version__}')

---
## 🌐 2. Definición del Universo de Activos

In [ ]:
# ── Universo de activos ─────────────────────────────────────────────────
UNIVERSE = {
    # Acciones individuales — sector semiconductores / mega-cap tech
    'individual': {
        'NVDA':  'NVIDIA Corporation',
        'MSFT':  'Microsoft Corporation',
        'GOOGL': 'Alphabet Inc. (Google)',
    },
    # ETFs temáticos
    'etf': {
        'SOXX':  'iShares Semiconductor ETF (Microprocesadores)',
        'SMH':   'VanEck Semiconductor ETF (Materiales Semiconductores)',
        'TAN':   'Invesco Solar ETF (Energía Solar)',
        'NLR':   'VanEck Uranium+Nuclear ETF (Energía Nuclear)',
        'URNM':  'Sprott Uranium Miners ETF (Uranio Puro)',
    },
    # Índices de referencia (benchmarks)
    'benchmark': {
        'SPY':   'SPDR S&P 500 ETF (benchmark amplio)',
        'QQQ':   'Invesco Nasdaq-100 ETF (tech benchmark)',
        '^GSPC': 'S&P 500 Index',
        '^IXIC': 'NASDAQ Composite Index',
    },
}

# Ticker plano para descargas masivas (excluye índices raw con ^)
ALL_TICKERS      = list(UNIVERSE['individual']) + list(UNIVERSE['etf']) + ['SPY', 'QQQ']
PORTFOLIO_TICKERS = list(UNIVERSE['individual']) + list(UNIVERSE['etf'])

# Ventana temporal por defecto
END_DATE   = datetime.today().strftime('%Y-%m-%d')
START_1Y   = (datetime.today() - relativedelta(years=1)).strftime('%Y-%m-%d')
START_3Y   = (datetime.today() - relativedelta(years=3)).strftime('%Y-%m-%d')
START_5Y   = (datetime.today() - relativedelta(years=5)).strftime('%Y-%m-%d')

print('📋 Universo de activos:')
for cat, assets in UNIVERSE.items():
    print(f'\n  [{cat.upper()}]')
    for ticker, name in assets.items():
        print(f'    {ticker:<8} {name}')

print(f'\n📅 Ventanas: 1Y={START_1Y} | 3Y={START_3Y} | 5Y={START_5Y} → {END_DATE}')

---
# 🏦 MÓDULO A — Macroeconómico
> Endpoints: A1 Inflación | A2 Tasas Fed | A5 Yield Curve

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# A. CARGA DE DATOS MACRO — FRED vía pandas-datareader
# ══════════════════════════════════════════════════════════════════════

FRED_SERIES = {
    # A1 — Inflación
    'CPI_YOY':        'CPIAUCSL',     # Consumer Price Index (All Urban)
    'CORE_CPI_YOY':   'CPILFESL',     # Core CPI (ex food & energy)
    'PPI_YOY':        'PPIACO',       # Producer Price Index
    # A2 — Tasas de interés
    'FED_FUNDS_RATE': 'FEDFUNDS',     # Federal Funds Effective Rate
    'SOFR':           'SOFR',         # Secured Overnight Financing Rate
    # A3 — Actividad económica
    'GDP_GROWTH':     'A191RL1Q225SBEA',  # Real GDP Growth QoQ annualized
    # A4 — Desempleo
    'UNEMPLOYMENT':   'UNRATE',       # Civilian Unemployment Rate
    'PARTICIPATION':  'CIVPART',      # Labor Force Participation Rate
    'NONFARM_PAY':    'PAYEMS',       # Total Nonfarm Payrolls (miles)
    # A5 — Yield Curve
    'UST_3M':  'DGS3MO',
    'UST_6M':  'DGS6MO',
    'UST_1Y':  'DGS1',
    'UST_2Y':  'DGS2',
    'UST_5Y':  'DGS5',
    'UST_10Y': 'DGS10',
    'UST_30Y': 'DGS30',
    # Sentimiento / Riesgo
    'VIX':    'VIXCLS',              # CBOE Volatility Index
}

macro_start = '2019-01-01'
macro_raw = {}

print('⏳ Descargando datos macro desde FRED...')
for label, fred_code in FRED_SERIES.items():
    try:
        series = web.DataReader(fred_code, 'fred', macro_start, END_DATE)
        macro_raw[label] = series[fred_code]
        print(f'  ✓ {label:<20} ({fred_code}) — {len(series)} observaciones')
    except Exception as e:
        print(f'  ✗ {label:<20} ({fred_code}) — Error: {e}')

macro = pd.DataFrame(macro_raw)
print(f'\n✅ DataFrame macro: {macro.shape[0]} filas × {macro.shape[1]} columnas')
macro.tail(3)

In [ ]:
# ── A1: Endpoint /api/v1/macro/inflation — Visualización ──────────────

def compute_inflation_yoy(series: pd.Series) -> pd.Series:
    """YoY % change para índices de precios (nivel, no tasa)."""
    return series.pct_change(12) * 100

cpi_yoy  = compute_inflation_yoy(macro['CPI_YOY'].dropna())
core_yoy = compute_inflation_yoy(macro['CORE_CPI_YOY'].dropna())
ppi_yoy  = compute_inflation_yoy(macro['PPI_YOY'].dropna())

# Fed target
target = 2.0

fig, axes = plt.subplots(2, 1, figsize=(15, 9), facecolor='#0e0e0e')

# Panel 1 — CPI, Core CPI, PPI YoY
ax = axes[0]
ax.set_facecolor('#0e0e0e')
ax.plot(cpi_yoy.index,  cpi_yoy.values,  label='CPI YoY (%)',      color='#ff4444', lw=2)
ax.plot(core_yoy.index, core_yoy.values, label='Core CPI YoY (%)', color='#ff9900', lw=2, ls='--')
ax.plot(ppi_yoy.index,  ppi_yoy.values,  label='PPI YoY (%)',      color='#aa88ff', lw=1.5, alpha=0.8)
ax.axhline(target, color='#00ff88', lw=1.5, ls=':', label=f'Fed Target ({target}%)')
ax.fill_between(cpi_yoy.index, target, cpi_yoy.values,
                where=(cpi_yoy.values > target),
                alpha=0.15, color='#ff4444', label='Above Target')
ax.set_title('A1 · Inflación USA (CPI / Core CPI / PPI)', fontsize=14, fontweight='bold', color='white')
ax.set_ylabel('Variación Anual (%)', color='white')
ax.legend(loc='upper left', framealpha=0.3)
ax.tick_params(colors='white')
ax.spines[['top','right']].set_visible(False)
for sp in ['bottom','left']:
    ax.spines[sp].set_color('#444')

# Panel 2 — Fed Funds Rate
ax2 = axes[1]
ax2.set_facecolor('#0e0e0e')
fed = macro['FED_FUNDS_RATE'].dropna()
ax2.fill_between(fed.index, 0, fed.values, alpha=0.4, color='#00aaff')
ax2.plot(fed.index, fed.values, label='Fed Funds Rate (%)', color='#00aaff', lw=2.5)
ax2.set_title('A2 · Tasa de la Reserva Federal (Fed Funds Rate)', fontsize=14, fontweight='bold', color='white')
ax2.set_ylabel('Tasa (%)', color='white')
ax2.legend(framealpha=0.3)
ax2.tick_params(colors='white')
ax2.spines[['top','right']].set_visible(False)
for sp in ['bottom','left']:
    ax2.spines[sp].set_color('#444')

plt.tight_layout()
plt.savefig('macro_inflation_rates.png', dpi=150, bbox_inches='tight', facecolor='#0e0e0e')
plt.show()
print(f'CPI actual: {cpi_yoy.iloc[-1]:.2f}% | Core CPI: {core_yoy.iloc[-1]:.2f}% | Fed Rate: {fed.iloc[-1]:.2f}%')

In [ ]:
# ── A5: Endpoint /api/v1/macro/yield-curve ─────────────────────────────

TENORS = ['UST_3M', 'UST_6M', 'UST_1Y', 'UST_2Y', 'UST_5Y', 'UST_10Y', 'UST_30Y']
TENOR_LABELS = ['3M', '6M', '1Y', '2Y', '5Y', '10Y', '30Y']

yield_df = macro[TENORS].dropna(how='all')
yield_df.columns = TENOR_LABELS

# Snapshots para comparar
snapshots = {
    'Hoy':       yield_df.iloc[-1],
    'Hace 3M':   yield_df.iloc[-63] if len(yield_df) > 63 else yield_df.iloc[0],
    'Hace 1 año':yield_df.iloc[-252] if len(yield_df) > 252 else yield_df.iloc[0],
}

# Spreads clave
spread_2y10y = (yield_df['10Y'] - yield_df['2Y']).dropna()
spread_3m10y = (yield_df['10Y'] - yield_df['3M']).dropna()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Yield Curve — Comparativa de Fechas', '2Y-10Y Spread (Indicador de Recesión)'],
)

palette = ['#00ff88', '#ffaa00', '#ff4444']
for (label, snap), color in zip(snapshots.items(), palette):
    fig.add_trace(go.Scatter(
        x=TENOR_LABELS, y=snap.values,
        mode='lines+markers',
        name=label,
        line=dict(color=color, width=2.5),
        marker=dict(size=8)
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=spread_2y10y.index, y=spread_2y10y.values,
    mode='lines', name='2Y-10Y Spread',
    line=dict(color='#00aaff', width=2),
    fill='tozeroy',
    fillcolor='rgba(0,100,200,0.2)'
), row=1, col=2)

fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=2)

fig.update_layout(
    template='plotly_dark',
    title='A5 · Yield Curve USA — Estructura de Tasas del Tesoro',
    height=450,
    showlegend=True,
)
fig.update_yaxes(title_text='Rendimiento (%)', row=1, col=1)
fig.update_yaxes(title_text='Spread (pp)', row=1, col=2)
fig.show()

print(f'\n📊 Yield Curve — Estado actual:')
print(f'  2Y-10Y Spread: {spread_2y10y.iloc[-1]:.3f}pp  {"🔴 INVERTIDA" if spread_2y10y.iloc[-1] < 0 else "🟢 NORMAL"}')
print(f'  3M-10Y Spread: {spread_3m10y.iloc[-1]:.3f}pp  {"🔴 INVERTIDA" if spread_3m10y.iloc[-1] < 0 else "🟢 NORMAL"}')
print(f'\n  Tasas actuales:')
print(yield_df.iloc[-1].to_string())

---
# 🏢 MÓDULO B — Microeconómico / Fundamental
> Endpoints: B4 Ratios | B5 Company Overview | B6 Earnings

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# B. CARGA DE DATOS FUNDAMENTALES — yfinance
# ══════════════════════════════════════════════════════════════════════

EQUITY_TICKERS = ['NVDA', 'MSFT', 'GOOGL']  # Solo acciones, no ETFs para fundamentales

fundamentals = {}
print('⏳ Descargando datos fundamentales (info, financials, earnings)...')

for ticker in EQUITY_TICKERS:
    t = yf.Ticker(ticker)
    fundamentals[ticker] = {
        'info':            t.info,
        'income_annual':   t.financials,
        'income_q':        t.quarterly_financials,
        'balance_annual':  t.balance_sheet,
        'balance_q':       t.quarterly_balance_sheet,
        'cashflow_annual': t.cashflow,
        'cashflow_q':      t.quarterly_cashflow,
        'earnings':        t.earnings_history,
        'recommendations': t.recommendations,
    }
    print(f'  ✓ {ticker} — {fundamentals[ticker]["info"].get("longName", "N/A")}')

print('\n✅ Fundamentales cargados.')

In [ ]:
# ── B4: Endpoint /api/v1/fundamental/ratios ────────────────────────────

def extract_ratios(ticker: str) -> dict:
    """Extrae y calcula ratios de valoración, rentabilidad y deuda desde yfinance."""
    info = fundamentals[ticker]['info']

    # yfinance entrega estos directamente (TTM)
    ratios = {
        'ticker':               ticker,
        'name':                 info.get('longName', ''),
        'sector':               info.get('sector', ''),
        'market_cap_B':         round(info.get('marketCap', 0) / 1e9, 2),
        # Valoración
        'pe_ratio':             info.get('trailingPE'),
        'forward_pe':           info.get('forwardPE'),
        'peg_ratio':            info.get('pegRatio'),
        'ps_ratio':             info.get('priceToSalesTrailing12Months'),
        'pb_ratio':             info.get('priceToBook'),
        'ev_ebitda':            info.get('enterpriseToEbitda'),
        'ev_revenue':           info.get('enterpriseToRevenue'),
        # Rentabilidad
        'roe':                  info.get('returnOnEquity'),
        'roa':                  info.get('returnOnAssets'),
        'gross_margin':         info.get('grossMargins'),
        'ebitda_margin':        info.get('ebitdaMargins'),
        'net_margin':           info.get('profitMargins'),
        'operating_margin':     info.get('operatingMargins'),
        # Crecimiento (YoY TTM)
        'rev_growth_yoy':       info.get('revenueGrowth'),
        'earnings_growth_yoy':  info.get('earningsGrowth'),
        # Deuda
        'debt_to_equity':       info.get('debtToEquity'),
        'current_ratio':        info.get('currentRatio'),
        'quick_ratio':          info.get('quickRatio'),
        # Dividendos / FCF
        'dividend_yield':       info.get('dividendYield'),
        'payout_ratio':         info.get('payoutRatio'),
        'fcf_B':                round(info.get('freeCashflow', 0) / 1e9, 2) if info.get('freeCashflow') else None,
        # Precio
        'current_price':        info.get('currentPrice'),
        '52w_high':             info.get('fiftyTwoWeekHigh'),
        '52w_low':              info.get('fiftyTwoWeekLow'),
        'beta':                 info.get('beta'),
        'analyst_target':       info.get('targetMeanPrice'),
    }
    return ratios

ratios_list = [extract_ratios(t) for t in EQUITY_TICKERS]
ratios_df = pd.DataFrame(ratios_list).set_index('ticker')

# ── Visualización: Tabla de ratios ────────────────────────────────────
display_cols = [
    'market_cap_B', 'pe_ratio', 'forward_pe', 'peg_ratio',
    'ev_ebitda', 'ps_ratio', 'pb_ratio',
    'roe', 'net_margin', 'gross_margin',
    'rev_growth_yoy', 'earnings_growth_yoy',
    'debt_to_equity', 'current_ratio', 'beta'
]

print('\n📊 B4 · Ratios Fundamentales — Comparativa')
print('=' * 80)
display(ratios_df[display_cols].T.style
    .format(lambda x: f'{x:.4f}' if isinstance(x, float) else str(x))
    .background_gradient(cmap='RdYlGn', axis=1)
)

In [ ]:
# ── B4: Radar Chart de Ratios de Rentabilidad ─────────────────────────

radar_metrics = {
    'ROE':          'roe',
    'ROA':          'roa',
    'Gross Margin': 'gross_margin',
    'Net Margin':   'net_margin',
    'Op. Margin':   'operating_margin',
    'Rev Growth':   'rev_growth_yoy',
    'EPS Growth':   'earnings_growth_yoy',
}

categories = list(radar_metrics.keys())
N = len(categories)

fig = go.Figure()
colors_radar = ['#76b900', '#00a4ef', '#fbbc05']

for ticker, color in zip(EQUITY_TICKERS, colors_radar):
    values = [
        ratios_df.loc[ticker, col] or 0
        for col in radar_metrics.values()
    ]
    values += [values[0]]  # cerrar polígono
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=ticker,
        line_color=color,
        fillcolor=color.replace('#', 'rgba(').replace(')', ',0.15)') if '#' in color else color,
        opacity=0.9,
    ))

fig.update_layout(
    template='plotly_dark',
    polar=dict(radialaxis=dict(visible=True, range=[-0.1, 1.0])),
    title='B4 · Perfil de Rentabilidad — NVDA vs MSFT vs GOOGL',
    showlegend=True,
    height=500,
)
fig.show()

In [ ]:
# ── B6: Endpoint /api/v1/fundamental/earnings ─────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='#0e0e0e')

for ax, ticker in zip(axes, EQUITY_TICKERS):
    ax.set_facecolor('#0e0e0e')
    color = list(COLORS.values())[EQUITY_TICKERS.index(ticker)]

    # Ingresos anuales (del income statement)
    inc = fundamentals[ticker]['income_annual']
    if inc is not None and not inc.empty:
        rev_row = 'Total Revenue'
        ni_row  = 'Net Income'
        if rev_row in inc.index:
            rev = inc.loc[rev_row].sort_index() / 1e9
            ax.bar(rev.index.year, rev.values, color=color, alpha=0.6, label='Revenue ($B)')
        if ni_row in inc.index:
            ni = inc.loc[ni_row].sort_index() / 1e9
            ax.plot(ni.index.year, ni.values, 'o--', color='white',
                    lw=2, ms=6, label='Net Income ($B)')

    ax.set_title(f'{ticker}', fontsize=13, fontweight='bold', color=color)
    ax.set_xlabel('Año Fiscal', color='#aaa')
    ax.set_ylabel('USD Billones ($B)', color='#aaa')
    ax.legend(framealpha=0.3, fontsize=9)
    ax.tick_params(colors='white')
    ax.spines[['top','right']].set_visible(False)
    for sp in ['bottom','left']:
        ax.spines[sp].set_color('#444')

plt.suptitle('B2/B6 · Ingresos y Utilidad Neta Anual', fontsize=15,
             fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

---
# 📈 MÓDULO C — Mercado / Técnico
> Endpoints: C1 Quote | C2 Historical OHLCV | C3 Volatilidad | C4 Indicadores Técnicos | C5 Sentimiento

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C. CARGA DE PRECIOS HISTÓRICOS — yfinance (C1 + C2)
# ══════════════════════════════════════════════════════════════════════

print('⏳ Descargando precios históricos (5 años, diarios, ajustados)...')

# Precio de cierre ajustado — todos los activos
raw_prices = yf.download(
    tickers=ALL_TICKERS,
    start=START_5Y,
    end=END_DATE,
    interval='1d',
    auto_adjust=True,
    progress=True,
)

# OHLCV separados
close  = raw_prices['Close'].dropna(how='all')
open_  = raw_prices['Open'].dropna(how='all')
high   = raw_prices['High'].dropna(how='all')
low    = raw_prices['Low'].dropna(how='all')
volume = raw_prices['Volume'].dropna(how='all')

# Retornos logarítmicos diarios
log_returns = np.log(close / close.shift(1)).dropna()
# Retornos simples diarios
simple_returns = close.pct_change().dropna()

print(f'\n✅ Precios: {close.shape[0]} días de trading × {close.shape[1]} activos')
print(f'   Desde: {close.index[0].date()} → {close.index[-1].date()}')
print('\n📊 Precios de cierre más recientes:')
display(close.tail(3).round(2))

In [ ]:
# ── C2: Performance Normalizada — todos los activos ───────────────────

close_1y = close.loc[START_1Y:]
normalized = (close_1y / close_1y.iloc[0]) * 100  # Base 100

fig = go.Figure()

for ticker in ALL_TICKERS:
    if ticker not in normalized.columns:
        continue
    color = COLORS.get(ticker, '#888888')
    lw = 2.5 if ticker in ['SPY', 'QQQ'] else 2
    dash = 'dot' if ticker in ['SPY', 'QQQ'] else 'solid'
    fig.add_trace(go.Scatter(
        x=normalized.index,
        y=normalized[ticker],
        name=ticker,
        line=dict(color=color, width=lw, dash=dash),
        hovertemplate='%{x|%Y-%m-%d}<br>%{y:.1f}<extra>' + ticker + '</extra>'
    ))

fig.add_hline(y=100, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(
    template='plotly_dark',
    title='C2 · Retorno Total Normalizado — Base 100 (1 Año)',
    yaxis_title='Índice (Base 100 al inicio)',
    xaxis_title='Fecha',
    height=520,
    hovermode='x unified',
    legend=dict(orientation='v', x=1.01, y=0.5),
)
fig.show()

In [ ]:
# ── C4: Endpoint /api/v1/market/indicators — Indicadores Técnicos ──────

def compute_technical_indicators(df_ohlcv: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    Calcula RSI-14, MACD(12,26,9), Bandas de Bollinger(20,2),
    ATR-14, SMA-50, SMA-200, EMA-21, OBV para un activo dado.
    """
    c = df_ohlcv['Close'][ticker].copy()
    h = df_ohlcv['High'][ticker].copy()
    l = df_ohlcv['Low'][ticker].copy()
    v = df_ohlcv['Volume'][ticker].copy()

    ind = pd.DataFrame(index=c.index)
    ind['Close'] = c

    # ── Momentum ─────────────────────────────────────────────────────
    ind['RSI_14']  = RSIIndicator(close=c, window=14).rsi()

    macd_obj = MACD(close=c, window_slow=26, window_fast=12, window_sign=9)
    ind['MACD']       = macd_obj.macd()
    ind['MACD_Signal'] = macd_obj.macd_signal()
    ind['MACD_Hist']  = macd_obj.macd_diff()

    # ── Volatilidad ──────────────────────────────────────────────────
    bb = BollingerBands(close=c, window=20, window_dev=2)
    ind['BB_Upper']  = bb.bollinger_hband()
    ind['BB_Middle'] = bb.bollinger_mavg()
    ind['BB_Lower']  = bb.bollinger_lband()
    ind['BB_PctB']   = bb.bollinger_pband()
    ind['BB_Width']  = bb.bollinger_wband()

    ind['ATR_14'] = AverageTrueRange(high=h, low=l, close=c, window=14).average_true_range()

    # ── Tendencia ────────────────────────────────────────────────────
    ind['SMA_50']  = SMAIndicator(close=c, window=50).sma_indicator()
    ind['SMA_200'] = SMAIndicator(close=c, window=200).sma_indicator()
    ind['EMA_21']  = EMAIndicator(close=c, window=21).ema_indicator()
    ind['ADX_14']  = ADXIndicator(high=h, low=l, close=c, window=14).adx()

    # ── Volumen ──────────────────────────────────────────────────────
    ind['OBV'] = OnBalanceVolumeIndicator(close=c, volume=v).on_balance_volume()

    # ── Señales derivadas ────────────────────────────────────────────
    ind['Golden_Cross']  = (ind['SMA_50'] > ind['SMA_200']).astype(int)
    ind['RSI_Signal']    = np.where(ind['RSI_14'] > 70, 'Overbought',
                            np.where(ind['RSI_14'] < 30, 'Oversold', 'Neutral'))
    ind['MACD_Crossover'] = np.where(ind['MACD'] > ind['MACD_Signal'], 'Bullish', 'Bearish')

    return ind

# Calcular para los 3 activos individuales
technicals = {}
for ticker in EQUITY_TICKERS:
    if ticker in close.columns:
        ohlcv_single = pd.concat({
            'Close': close[[ticker]],
            'High':  high[[ticker]],
            'Low':   low[[ticker]],
            'Volume': volume[[ticker]],
        }, axis=1)
        technicals[ticker] = compute_technical_indicators(ohlcv_single, ticker)

print('✅ Indicadores técnicos calculados para:', list(technicals.keys()))

# Resumen de señales actuales
print('\n📊 C4 · Señales Técnicas — Estado Actual:')
print(f'{"Ticker":<8} {"Precio":>10} {"RSI-14":>8} {"Señal RSI":<12} {"MACD Cross":<12} {"Golden X":>10} {"BB %B":>8}')
print('-' * 75)
for ticker, ind in technicals.items():
    last = ind.iloc[-1]
    gx = '✅ Sí' if last['Golden_Cross'] == 1 else '❌ No'
    print(f'{ticker:<8} {last["Close"]:>10.2f} {last["RSI_14"]:>8.1f} {last["RSI_Signal"]:<12} {last["MACD_Crossover"]:<12} {gx:>10} {last["BB_PctB"]:>8.3f}')

In [ ]:
# ── C4: Dashboard técnico completo para un activo ─────────────────────

def plot_technical_dashboard(ticker: str, lookback: int = 180):
    """Gráfico de 4 paneles: Precio+MAs+BB | RSI | MACD | Volumen."""
    ind = technicals[ticker].iloc[-lookback:].copy()

    fig, axes = plt.subplots(4, 1, figsize=(16, 14), facecolor='#0d0d0d',
                              gridspec_kw={'height_ratios': [3, 1, 1, 1]})
    color = COLORS.get(ticker, '#00aaff')

    # ── Panel 1: Precio + MAs + Bandas de Bollinger ──────────────────
    ax = axes[0]
    ax.set_facecolor('#0d0d0d')
    ax.fill_between(ind.index, ind['BB_Lower'], ind['BB_Upper'],
                    alpha=0.1, color='#4488ff', label='Bollinger Bands')
    ax.plot(ind.index, ind['BB_Upper'],  color='#4488ff', lw=0.8, ls='--', alpha=0.7)
    ax.plot(ind.index, ind['BB_Lower'],  color='#4488ff', lw=0.8, ls='--', alpha=0.7)
    ax.plot(ind.index, ind['BB_Middle'], color='#4488ff', lw=1, alpha=0.5, label='SMA-20')
    ax.plot(ind.index, ind['Close'],     color=color, lw=2, label=f'{ticker} Price')
    ax.plot(ind.index, ind['SMA_50'],    color='#ffaa00', lw=1.5, ls='--', label='SMA-50')
    ax.plot(ind.index, ind['SMA_200'],   color='#ff4444', lw=1.5, ls='--', label='SMA-200')
    ax.plot(ind.index, ind['EMA_21'],    color='#00ff88', lw=1, ls=':', label='EMA-21')

    # Marcar Golden/Death Cross
    cross_change = ind['Golden_Cross'].diff()
    for date, val in cross_change.items():
        if val == 1:
            ax.axvline(date, color='#00ff88', lw=1.5, alpha=0.6)
        elif val == -1:
            ax.axvline(date, color='#ff4444', lw=1.5, alpha=0.6)

    ax.set_title(f'{ticker} — Dashboard Técnico Completo ({lookback}d)', fontsize=14,
                 fontweight='bold', color='white')
    ax.legend(loc='upper left', framealpha=0.2, fontsize=9)
    ax.set_ylabel('Precio (USD)', color='#aaa')
    ax.tick_params(colors='white')

    # ── Panel 2: RSI ─────────────────────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor('#0d0d0d')
    ax2.plot(ind.index, ind['RSI_14'], color='#cc88ff', lw=2)
    ax2.axhline(70, color='#ff4444', lw=1, ls='--')
    ax2.axhline(30, color='#00ff88', lw=1, ls='--')
    ax2.axhline(50, color='#888', lw=0.5, ls=':')
    ax2.fill_between(ind.index, 70, ind['RSI_14'], where=(ind['RSI_14'] >= 70),
                     alpha=0.3, color='#ff4444')
    ax2.fill_between(ind.index, 30, ind['RSI_14'], where=(ind['RSI_14'] <= 30),
                     alpha=0.3, color='#00ff88')
    ax2.set_ylim(0, 100)
    ax2.set_ylabel('RSI-14', color='#aaa')
    ax2.tick_params(colors='white')

    # ── Panel 3: MACD ────────────────────────────────────────────────
    ax3 = axes[2]
    ax3.set_facecolor('#0d0d0d')
    colors_hist = ['#00ff88' if v >= 0 else '#ff4444' for v in ind['MACD_Hist']]
    ax3.bar(ind.index, ind['MACD_Hist'], color=colors_hist, alpha=0.7, width=1)
    ax3.plot(ind.index, ind['MACD'],        color='#00aaff', lw=1.5, label='MACD')
    ax3.plot(ind.index, ind['MACD_Signal'], color='#ff8800', lw=1.5, label='Signal')
    ax3.axhline(0, color='#888', lw=0.5)
    ax3.set_ylabel('MACD(12,26,9)', color='#aaa')
    ax3.legend(framealpha=0.2, fontsize=9, loc='upper left')
    ax3.tick_params(colors='white')

    # ── Panel 4: Volumen ─────────────────────────────────────────────
    ax4 = axes[3]
    ax4.set_facecolor('#0d0d0d')
    vol_colors = ['#00ff88' if c >= o else '#ff4444'
                  for c, o in zip(ind['Close'], ind['Close'].shift(1).fillna(method='bfill'))]
    if ticker in volume.columns:
        vol_data = volume[ticker].iloc[-lookback:]
        ax4.bar(vol_data.index, vol_data.values / 1e6, color=vol_colors, alpha=0.8, width=1)
        vol_ma = vol_data.rolling(20).mean() / 1e6
        ax4.plot(vol_ma.index, vol_ma.values, color='#ffaa00', lw=1.5, label='Vol MA-20')
    ax4.set_ylabel('Volumen (M)', color='#aaa')
    ax4.legend(framealpha=0.2, fontsize=9)
    ax4.tick_params(colors='white')

    for ax in axes:
        ax.spines[['top','right']].set_visible(False)
        for sp in ['bottom','left']:
            ax.spines[sp].set_color('#333')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    plt.tight_layout()
    plt.savefig(f'technical_{ticker}.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
    plt.show()

# Generar dashboard para cada acción individual
for ticker in EQUITY_TICKERS:
    if ticker in technicals:
        plot_technical_dashboard(ticker, lookback=180)

In [ ]:
# ── C3: Endpoint /api/v1/market/volatility — Volatilidad Realizada ─────

def compute_realized_volatility(returns: pd.DataFrame, windows: list = [10, 21, 63, 126, 252]) -> pd.DataFrame:
    """Volatilidad realizada anualizada para múltiples ventanas."""
    vols = {}
    for w in windows:
        vols[f'RV_{w}d'] = returns.rolling(w).std() * np.sqrt(252) * 100
    return pd.concat(vols, axis=1)

# Métricas de riesgo: Beta, Sharpe, Max Drawdown, VaR
def compute_risk_metrics(returns: pd.DataFrame, benchmark: str = 'SPY',
                         rf_annual: float = 0.0525) -> pd.DataFrame:
    rf_daily = rf_annual / 252
    bmark = returns[benchmark] if benchmark in returns.columns else returns.iloc[:, 0]

    results = []
    for col in returns.columns:
        r = returns[col].dropna()
        excess = r - rf_daily

        # Beta vs SPY (regresión)
        joint = pd.concat([r, bmark], axis=1).dropna()
        if len(joint) > 30:
            X = sm.add_constant(joint.iloc[:, 1])
            beta = sm.OLS(joint.iloc[:, 0], X).fit().params.iloc[1]
            corr = joint.corr().iloc[0, 1]
        else:
            beta, corr = np.nan, np.nan

        # Drawdown máximo
        cum = (1 + r).cumprod()
        roll_max = cum.cummax()
        drawdowns = (cum - roll_max) / roll_max
        max_dd = drawdowns.min()

        # VaR y CVaR 95%
        var_95  = np.percentile(r, 5)
        cvar_95 = r[r <= var_95].mean()

        # Volatilidad anualizada
        vol_ann = r.std() * np.sqrt(252)

        # Sharpe y Sortino
        ann_return = (1 + r.mean()) ** 252 - 1
        sharpe = (ann_return - rf_annual) / vol_ann if vol_ann > 0 else np.nan
        downside = r[r < 0].std() * np.sqrt(252)
        sortino = (ann_return - rf_annual) / downside if downside > 0 else np.nan

        results.append({
            'Ticker':          col,
            'Ann. Return (%)': round(ann_return * 100, 2),
            'Ann. Vol (%)':    round(vol_ann * 100, 2),
            'Sharpe':          round(sharpe, 3),
            'Sortino':         round(sortino, 3),
            'Max DD (%)':      round(max_dd * 100, 2),
            f'Beta vs {benchmark}': round(beta, 3),
            f'Corr vs {benchmark}': round(corr, 3),
            'VaR 95% (1d)':    round(var_95 * 100, 3),
            'CVaR 95% (1d)':   round(cvar_95 * 100, 3),
        })

    return pd.DataFrame(results).set_index('Ticker')

# Calcular para ventana de 1 año
returns_1y = simple_returns.loc[START_1Y:]
risk_metrics = compute_risk_metrics(returns_1y, benchmark='SPY')

print('📊 C3 · Métricas de Riesgo — 1 Año')
print('=' * 90)
display(risk_metrics.style
    .background_gradient(cmap='RdYlGn', subset=['Ann. Return (%)','Sharpe','Sortino'])
    .background_gradient(cmap='RdYlGn_r', subset=['Ann. Vol (%)','Max DD (%)'])
    .format('{:.3f}')
)

In [ ]:
# ── C3: Heatmap de Correlaciones + Volatilidad Rolling ────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor='#0e0e0e')

# Panel 1 — Correlación entre activos (1 año)
corr_matrix = returns_1y[PORTFOLIO_TICKERS].dropna(how='all', axis=1).corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, ax=axes[0], mask=~mask,
    annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, linecolor='#222',
    annot_kws={'size': 9},
    cbar_kws={'shrink': 0.8},
)
axes[0].set_title('Matriz de Correlaciones (1Y)', fontsize=13, fontweight='bold', color='white')
axes[0].set_facecolor('#0e0e0e')
axes[0].tick_params(colors='white')

# Panel 2 — Volatilidad Rolling 21d comparada
rv_21d = returns_1y.rolling(21).std() * np.sqrt(252) * 100

ax = axes[1]
ax.set_facecolor('#0e0e0e')
for ticker in PORTFOLIO_TICKERS:
    if ticker in rv_21d.columns:
        lw = 2.5 if ticker in EQUITY_TICKERS else 1.5
        ax.plot(rv_21d.index, rv_21d[ticker], label=ticker,
                color=COLORS.get(ticker, '#888'), lw=lw)

ax.set_title('Volatilidad Realizada Rolling 21d (Anualizada %)', fontsize=13,
             fontweight='bold', color='white')
ax.set_ylabel('Volatilidad Anualizada (%)', color='#aaa')
ax.legend(framealpha=0.2, fontsize=9)
ax.tick_params(colors='white')
ax.spines[['top','right']].set_visible(False)
for sp in ['bottom','left']:
    ax.spines[sp].set_color('#333')

plt.suptitle('C3 · Análisis de Correlación y Volatilidad', fontsize=15,
             fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.show()

---
# 💼 MÓDULO D — Portafolio / Optimización
> Endpoints: D1 Screening | D2 Análisis de Portafolio | D3 Optimización Markowitz

In [ ]:
# ── D1: Endpoint /api/v1/screening/stocks — Screener Fundamental ───────

# Construimos el screener sobre los activos individuales con datos disponibles
SCREENER_FILTERS = {
    'pe_ratio':         ('<=', 80),   # P/E alto permitido por crecimiento
    'forward_pe':       ('<=', 60),
    'roe':              ('>=', 0.15),  # ROE mínimo 15%
    'net_margin':       ('>=', 0.10),  # Margen neto mínimo 10%
    'gross_margin':     ('>=', 0.40),
    'rev_growth_yoy':   ('>=', 0.05),
    'debt_to_equity':   ('<=', 3.0),
    'current_ratio':    ('>=', 1.0),
    'beta':             ('<=', 2.5),
}

def apply_screener(df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for col, (op, val) in filters.items():
        if col not in df.columns:
            continue
        col_vals = pd.to_numeric(df[col], errors='coerce')
        if op == '<=':
            mask &= (col_vals <= val) | col_vals.isna()
        elif op == '>=':
            mask &= (col_vals >= val) | col_vals.isna()
        elif op == '==':
            mask &= (col_vals == val)
    return df[mask]

screener_results = apply_screener(ratios_df, SCREENER_FILTERS)

print(f'📊 D1 · Screener — Activos que pasan los filtros: {len(screener_results)}/{len(ratios_df)}')
print(f'   Filtros aplicados: {list(SCREENER_FILTERS.keys())}\n')

score_cols = ['pe_ratio', 'forward_pe', 'roe', 'net_margin', 'gross_margin',
              'rev_growth_yoy', 'debt_to_equity', 'ev_ebitda', 'beta']
display(ratios_df[score_cols].style
    .apply(lambda col: ['background-color: #1a3a1a' if idx in screener_results.index else ''
                        for idx in col.index], axis=0)
    .format('{:.4f}')
)

In [ ]:
# ── D2: Endpoint /api/v1/portfolio/analyze — Análisis de Portafolio ────

# Portafolio hipotético (pesos iguales entre todos los activos del universo)
PORTFOLIO_WEIGHTS_EQUAL = {
    t: 1/len(PORTFOLIO_TICKERS) for t in PORTFOLIO_TICKERS
    if t in close.columns
}

# Portafolio concentrado en tech (caso de estudio)
PORTFOLIO_WEIGHTS_TECH = {
    'NVDA':  0.30,
    'MSFT':  0.25,
    'GOOGL': 0.20,
    'SOXX':  0.15,
    'SMH':   0.10,
}

def portfolio_returns(weights: dict, returns_df: pd.DataFrame) -> pd.Series:
    valid = {k: v for k, v in weights.items() if k in returns_df.columns}
    total = sum(valid.values())
    normalized = {k: v/total for k, v in valid.items()}
    return returns_df[list(normalized.keys())].mul(pd.Series(normalized)).sum(axis=1)

def portfolio_stats(port_rets: pd.Series, bench_rets: pd.Series,
                    rf_annual: float = 0.0525, label: str = 'Portfolio') -> dict:
    rf_daily = rf_annual / 252
    ann_ret  = (1 + port_rets.mean()) ** 252 - 1
    ann_vol  = port_rets.std() * np.sqrt(252)
    sharpe   = (ann_ret - rf_annual) / ann_vol
    downside = port_rets[port_rets < rf_daily].std() * np.sqrt(252)
    sortino  = (ann_ret - rf_annual) / downside if downside > 0 else np.nan
    cum = (1 + port_rets).cumprod()
    max_dd = ((cum - cum.cummax()) / cum.cummax()).min()

    # Alpha y Beta vs benchmark
    joint = pd.concat([port_rets, bench_rets], axis=1).dropna()
    X = sm.add_constant(joint.iloc[:, 1])
    reg = sm.OLS(joint.iloc[:, 0], X).fit()
    alpha_ann = reg.params['const'] * 252
    beta = reg.params.iloc[1]

    return {
        'Label':             label,
        'Ann. Return (%)':   round(ann_ret * 100, 2),
        'Ann. Vol (%)':      round(ann_vol * 100, 2),
        'Sharpe':            round(sharpe, 3),
        'Sortino':           round(sortino, 3),
        'Max DD (%)':        round(max_dd * 100, 2),
        'Alpha (ann %)':     round(alpha_ann * 100, 2),
        'Beta vs SPY':       round(beta, 3),
        'Calmar Ratio':      round(-ann_ret / max_dd, 3) if max_dd != 0 else np.nan,
    }

rets_2y = simple_returns.loc[START_3Y:]
bench_rets = rets_2y['SPY'].dropna()

port_equal_rets = portfolio_returns(PORTFOLIO_WEIGHTS_EQUAL, rets_2y)
port_tech_rets  = portfolio_returns(PORTFOLIO_WEIGHTS_TECH, rets_2y)
spy_rets = rets_2y['SPY'].dropna()

stats_comparison = pd.DataFrame([
    portfolio_stats(port_equal_rets, bench_rets, label='Equal-Weight (8 activos)'),
    portfolio_stats(port_tech_rets,  bench_rets, label='Tech Concentrado (5 activos)'),
    portfolio_stats(spy_rets,        bench_rets, label='Benchmark SPY'),
]).set_index('Label')

print('📊 D2 · Análisis Comparativo de Portafolios (3 años)')
print('=' * 80)
display(stats_comparison.style
    .background_gradient(cmap='RdYlGn', subset=['Ann. Return (%)','Sharpe','Sortino','Alpha (ann %)'])
    .background_gradient(cmap='RdYlGn_r', subset=['Ann. Vol (%)','Max DD (%)'])
    .format('{:.3f}')
)

In [ ]:
# ── D2: Visualización — Curvas de Capital y Drawdown ──────────────────

fig, axes = plt.subplots(2, 1, figsize=(16, 10), facecolor='#0d0d0d')

port_cum = {
    'Equal-Weight':        (1 + port_equal_rets).cumprod() * 100,
    'Tech Concentrado':    (1 + port_tech_rets).cumprod() * 100,
    'SPY (Benchmark)':     (1 + spy_rets).cumprod() * 100,
    'QQQ':                 (1 + rets_2y['QQQ']).cumprod() * 100,
}
port_colors = ['#00ff88', '#76b900', '#1f77b4', '#9467bd']

# Panel 1: Curva de capital
ax = axes[0]
ax.set_facecolor('#0d0d0d')
for (name, cum), color in zip(port_cum.items(), port_colors):
    lw = 2.5 if name in ['Tech Concentrado', 'SPY (Benchmark)'] else 1.8
    ax.plot(cum.index, cum.values, label=name, color=color, lw=lw)
ax.axhline(100, color='#555', lw=1, ls='--')
ax.set_title('D2 · Curva de Capital Acumulado — Base 100', fontsize=13, fontweight='bold', color='white')
ax.set_ylabel('Capital ($)', color='#aaa')
ax.legend(framealpha=0.2)
ax.tick_params(colors='white')
ax.spines[['top','right']].set_visible(False)
for sp in ['bottom','left']:
    ax.spines[sp].set_color('#333')

# Panel 2: Drawdown
ax2 = axes[1]
ax2.set_facecolor('#0d0d0d')
for (name, cum), color in zip(port_cum.items(), port_colors):
    dd = (cum - cum.cummax()) / cum.cummax() * 100
    ax2.fill_between(dd.index, 0, dd.values, alpha=0.3, color=color)
    ax2.plot(dd.index, dd.values, color=color, lw=1.2, label=name)
ax2.set_title('Drawdown Relativo al Máximo (%)', fontsize=12, color='white')
ax2.set_ylabel('Drawdown (%)', color='#aaa')
ax2.legend(framealpha=0.2, fontsize=9)
ax2.tick_params(colors='white')
ax2.spines[['top','right']].set_visible(False)
for sp in ['bottom','left']:
    ax2.spines[sp].set_color('#333')

plt.tight_layout()
plt.show()

In [ ]:
# ── D3: Endpoint /api/v1/portfolio/optimize — Optimización Markowitz ───

# Datos: 3 años de retornos para todos los activos del portafolio
opt_returns = simple_returns[PORTFOLIO_TICKERS].loc[START_3Y:].dropna(how='all', axis=1)
available_tickers = opt_returns.columns.tolist()
opt_returns = opt_returns.dropna()

print(f'🔧 Optimizando portafolio con: {available_tickers}')
print(f'   Período: {opt_returns.index[0].date()} → {opt_returns.index[-1].date()} ({len(opt_returns)} días)\n')

# ── Inputs de PyPortfolioOpt ─────────────────────────────────────────
mu    = expected_returns.mean_historical_return(opt_returns, returns_data=True)
sigma = risk_models.sample_cov(opt_returns, returns_data=True)

# ── Estrategia 1: Máximo Sharpe ───────────────────────────────────────
ef_sharpe = EfficientFrontier(mu, sigma, weight_bounds=(0.02, 0.40))
ef_sharpe.add_objective(objective_functions.L2_reg, gamma=0.1)  # evitar concentración
raw_sharpe = ef_sharpe.max_sharpe(risk_free_rate=0.0525)
weights_sharpe = ef_sharpe.clean_weights()
perf_sharpe = ef_sharpe.portfolio_performance(verbose=False, risk_free_rate=0.0525)

# ── Estrategia 2: Mínima Volatilidad ─────────────────────────────────
ef_minvol = EfficientFrontier(mu, sigma, weight_bounds=(0.02, 0.40))
ef_minvol.min_volatility()
weights_minvol = ef_minvol.clean_weights()
perf_minvol = ef_minvol.portfolio_performance(verbose=False, risk_free_rate=0.0525)

# ── Estrategia 3: Risk Parity (HRP) ──────────────────────────────────
from pypfopt import HRPOpt
hrp = HRPOpt(opt_returns)
hrp.optimize()
weights_hrp = hrp.clean_weights()
hrp_rets = portfolio_returns(weights_hrp, opt_returns)
hrp_ann_ret = (1 + hrp_rets.mean()) ** 252 - 1
hrp_ann_vol = hrp_rets.std() * np.sqrt(252)
perf_hrp = (hrp_ann_ret, hrp_ann_vol,
            (hrp_ann_ret - 0.0525) / hrp_ann_vol)

# ── Resumen de estrategias ────────────────────────────────────────────
strategies = {
    'Max Sharpe':     (weights_sharpe, perf_sharpe),
    'Min Volatility': (weights_minvol, perf_minvol),
    'HRP (Risk Parity)': (weights_hrp, perf_hrp),
}

print('📊 D3 · Estrategias de Optimización — Resultados:')
print('─' * 70)
for name, (weights, perf) in strategies.items():
    ret, vol, sharpe = perf
    print(f'\n  {name}:')
    print(f'    Expected Return: {ret*100:.2f}%')
    print(f'    Ann. Volatility: {vol*100:.2f}%')
    print(f'    Sharpe Ratio:    {sharpe:.3f}')
    non_zero = {k: v for k, v in weights.items() if v > 0.001}
    print(f'    Pesos: {non_zero}')

In [ ]:
# ── D3: Frontera Eficiente + portfolios óptimos ───────────────────────

# Simular 10,000 portafolios aleatorios para visualizar la frontera
np.random.seed(42)
n_simulations = 10_000
n_assets = len(available_tickers)

sim_returns = []
sim_vols    = []
sim_sharpes = []
sim_weights = []

ann_cov = sigma.values
ann_mu  = mu.values

for _ in range(n_simulations):
    w = np.random.dirichlet(np.ones(n_assets))
    r = ann_mu @ w
    v = np.sqrt(w @ ann_cov @ w)
    s = (r - 0.0525) / v
    sim_returns.append(r)
    sim_vols.append(v)
    sim_sharpes.append(s)
    sim_weights.append(w)

sim_returns = np.array(sim_returns)
sim_vols    = np.array(sim_vols)
sim_sharpes = np.array(sim_sharpes)

fig = go.Figure()

# Nube de portafolios simulados
fig.add_trace(go.Scatter(
    x=sim_vols * 100, y=sim_returns * 100,
    mode='markers',
    marker=dict(
        size=3, opacity=0.4,
        color=sim_sharpes, colorscale='Viridis',
        colorbar=dict(title='Sharpe Ratio'),
        showscale=True
    ),
    name='Portafolios Simulados',
    hovertemplate='Vol: %{x:.1f}%<br>Ret: %{y:.1f}%<extra></extra>'
))

# Marcar los 3 portafolios óptimos
optimal_markers = [
    ('Max Sharpe',      perf_sharpe, '⭐', '#ffdd00', 20),
    ('Min Volatility',  perf_minvol, '🛡️', '#00ff88', 20),
    ('HRP Risk Parity', perf_hrp,    '⚖️', '#ff8800', 18),
]

for name, perf, symbol, color, size in optimal_markers:
    ret, vol, sharpe = perf
    fig.add_trace(go.Scatter(
        x=[vol * 100], y=[ret * 100],
        mode='markers+text',
        marker=dict(size=size, color=color, symbol='star',
                    line=dict(width=2, color='white')),
        text=[name],
        textposition='top right',
        textfont=dict(color=color, size=11),
        name=f'{name} (Sharpe={sharpe:.2f})',
    ))

# Capital Market Line (CML)
rf = 0.0525
sharpe_opt_ret, sharpe_opt_vol, _ = perf_sharpe
slope = (sharpe_opt_ret - rf) / sharpe_opt_vol
vol_range = np.linspace(0, sharpe_opt_vol * 1.5, 50)
cml_line = rf + slope * vol_range

fig.add_trace(go.Scatter(
    x=vol_range * 100, y=cml_line * 100,
    mode='lines',
    line=dict(color='white', dash='dash', width=1.5),
    name='Capital Market Line',
))

fig.add_scatter(x=[0], y=[rf * 100], mode='markers+text',
                marker=dict(size=10, color='white'),
                text=['Risk-Free Rate'], textposition='bottom right',
                textfont=dict(color='white', size=10),
                name=f'Risk-Free ({rf*100:.2f}%)', showlegend=True)

fig.update_layout(
    template='plotly_dark',
    title='D3 · Frontera Eficiente de Markowitz — 10,000 Portafolios Simulados',
    xaxis_title='Volatilidad Anualizada (%)',
    yaxis_title='Retorno Esperado Anualizado (%)',
    height=600,
    showlegend=True,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(0,0,0,0.5)'),
)
fig.show()

In [ ]:
# ── D3: Comparación de Pesos por Estrategia ────────────────────────────

weights_df = pd.DataFrame({
    'Max Sharpe':      dict(weights_sharpe),
    'Min Volatility':  dict(weights_minvol),
    'HRP Risk Parity': dict(weights_hrp),
    'Equal Weight':    {t: 1/len(available_tickers) for t in available_tickers},
}).T

# Filtrar solo columnas con datos
weights_df = weights_df[[c for c in available_tickers if c in weights_df.columns]]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='#0d0d0d')
axes = axes.flatten()
strategy_names = weights_df.index.tolist()
strategy_colors = ['#ffdd00', '#00ff88', '#ff8800', '#00aaff']

for ax, (strat, color) in zip(axes, zip(strategy_names, strategy_colors)):
    ax.set_facecolor('#0d0d0d')
    row = weights_df.loc[strat]
    non_zero = row[row > 0.001].sort_values(ascending=False)
    ticker_colors = [COLORS.get(t, '#888') for t in non_zero.index]

    bars = ax.bar(non_zero.index, non_zero.values * 100, color=ticker_colors, alpha=0.85,
                  edgecolor='white', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.3,
                f'{h:.1f}%', ha='center', va='bottom', color='white', fontsize=9)

    ax.set_title(strat, fontsize=12, fontweight='bold', color=color)
    ax.set_ylabel('Peso (%)', color='#aaa')
    ax.tick_params(axis='x', rotation=45, colors='white')
    ax.tick_params(axis='y', colors='white')
    ax.spines[['top','right']].set_visible(False)
    for sp in ['bottom','left']:
        ax.spines[sp].set_color('#333')

plt.suptitle('D3 · Distribución de Pesos por Estrategia de Optimización',
             fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.show()

print('\n📊 Tabla de Pesos:')
display((weights_df * 100).round(2).style
    .format('{:.2f}%')
    .background_gradient(cmap='YlOrRd', axis=1)
)

---
# 📋 Dashboard Ejecutivo — Resumen Final

In [ ]:
# ── Dashboard Ejecutivo: Vista de 1 página ─────────────────────────────

print('═' * 80)
print('  FINANCIAL ANALYSIS PLATFORM — RESUMEN EJECUTIVO')
print(f'  Generado: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} UTC')
print('═' * 80)

print('\n【MACRO CONTEXTO】')
fed_rate = macro['FED_FUNDS_RATE'].dropna().iloc[-1]
cpi_cur  = cpi_yoy.iloc[-1]
spread   = spread_2y10y.iloc[-1]
vix_cur  = macro['VIX'].dropna().iloc[-1]
print(f'  Fed Funds Rate:  {fed_rate:.2f}%')
print(f'  CPI YoY:         {cpi_cur:.2f}%  (Target: 2.00%)')
print(f'  Yield 2Y-10Y:    {spread:.3f}pp  {"🔴 INVERTIDA" if spread < 0 else "🟢 NORMAL"}')
print(f'  VIX:             {vix_cur:.2f}  {"⚠️ Alto" if vix_cur > 25 else "✅ Bajo"}')

print('\n【ACCIONES INDIVIDUALES — SEÑALES】')
for ticker in EQUITY_TICKERS:
    if ticker not in technicals or ticker not in close.columns:
        continue
    last   = technicals[ticker].iloc[-1]
    price  = last['Close']
    rsi    = last['RSI_14']
    macd_x = last['MACD_Crossover']
    gx     = '✅' if last['Golden_Cross'] == 1 else '❌'
    target = ratios_df.loc[ticker, 'analyst_target'] if ticker in ratios_df.index else None
    upside = f"{((target/price)-1)*100:.1f}%" if target and target > 0 else 'N/A'
    print(f'  {ticker:<6} | Precio: ${price:>8.2f} | RSI: {rsi:>5.1f} | '
          f'MACD: {macd_x:<8} | Golden Cross: {gx} | Potencial: {upside}')

print('\n【ETFs TEMÁTICOS — PERFORMANCE 1Y】')
for ticker in list(UNIVERSE['etf'].keys()):
    if ticker not in close.columns:
        continue
    close_1y_t = close[ticker].loc[START_1Y:].dropna()
    if len(close_1y_t) < 2:
        continue
    ret_1y = (close_1y_t.iloc[-1] / close_1y_t.iloc[0] - 1) * 100
    vol_1y = simple_returns[ticker].loc[START_1Y:].std() * np.sqrt(252) * 100
    sign = '📈' if ret_1y > 0 else '📉'
    print(f'  {ticker:<6} {UNIVERSE["etf"][ticker][:40]:<42} | {sign} {ret_1y:>+7.1f}% | Vol: {vol_1y:.1f}%')

print('\n【OPTIMIZACIÓN DE PORTAFOLIO】')
for name, (weights, perf) in strategies.items():
    ret, vol, sharpe = perf
    print(f'  {name:<22} | Ret: {ret*100:>6.2f}% | Vol: {vol*100:>5.2f}% | Sharpe: {sharpe:>5.3f}')

print('\n【RECOMENDACIÓN】')
best_strat = max(strategies.items(), key=lambda x: x[1][1][2])
print(f'  Estrategia óptima por Sharpe: {best_strat[0]}')
print(f'  Top 3 pesos recomendados:')
top3 = sorted(best_strat[1][0].items(), key=lambda x: x[1], reverse=True)[:3]
for t, w in top3:
    print(f'    {t:<6} → {w*100:.1f}%')

print('\n' + '═' * 80)
print('  ⚠️  DISCLAIMER: Este análisis es exclusivamente educativo.')
print('      No constituye asesoramiento financiero ni recomendación de inversión.')
print('═' * 80)